**Studi Kasus:**

Analisis Dashboard Performa Cabang E-Commerce & Perbandingan Target Penjualan  

In [1]:
import os
import numpy as np
import pandas as pd

# Tabel 1: Target & PIC per cabang kota (tabel referensi, dibuat langsung sebagai DataFrame)
data_target_cabang = {
    "kota": ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"],
    "target_bulanan": [45000000, 60000000, 55000000, 40000000, 30000000],
    "pic_cabang": ["Rani", "Joko", "Sari", "Bayu", "Fitri"],
}

# Tabel 2: Data transaksi (disimpan sebagai CSV, lalu diunggah ke HDFS)
np.random.seed(55)
n = 500
kategori_list = ["Elektronik", "Fashion", "Makanan & Minuman", "Kesehatan & Kecantikan", "Rumah Tangga"]
kota_list = ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"]
data_transaksi_t5 = {
    "order_id": [f"TRX-{i}" for i in range(n)],
    "kategori": np.random.choice(kategori_list, size=n),
    "kota": np.random.choice(kota_list, size=n),
    "unit_terjual": np.random.randint(1, 10, size=n),
    "harga_satuan": np.random.choice([25000, 50000, 75000, 100000, 150000], size=n),
}

df_t5_local = pd.DataFrame(data_transaksi_t5)
df_t5_local.to_csv("transaksi_tugas5.csv", index=False)
print(f"Dataset lokal berhasil dibuat: {df_t5_local.shape[0]} baris.")

# Mengunggah ke klaster HDFS menggunakan variabel dinamis $USER (Arch Linux standard)
!hdfs dfs -mkdir -p /user/$USER/tugas5
!hdfs dfs -put -f transaksi_tugas5.csv /user/$USER/tugas5/
username = os.environ.get("USER", "kaky")
print(f"Dataset siap. Tabel transaksi sudah diunggah ke HDFS: /user/{username}/tugas5/transaksi_tugas5.csv")

Dataset lokal berhasil dibuat: 500 baris.


Dataset siap. Tabel transaksi sudah diunggah ke HDFS: /user/kaky/tugas5/transaksi_tugas5.csv


In [3]:
import sys
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Konfigurasi isolasi lingkungan native Arch Linux
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk"
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

# Buat SparkSession
spark = SparkSession.builder \
    .appName("TugasMandiri5-ArchLinux") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("SparkSession berhasil diinisialisasi!")

SparkSession berhasil diinisialisasi!


In [5]:
# 1. Baca transaksi_tugas5.csv dari HDFS dan tambahkan kolom pendapatan
hdfs_path = f"hdfs://localhost:9000/user/{username}/tugas5/transaksi_tugas5.csv"

df_transaksi = spark.read.option("header", "true") \
    .option("inferSchema", "true") \
    .csv(hdfs_path)

# Tambahkan kolom pendapatan (unit_terjual x harga_satuan)
df_transaksi = df_transaksi.withColumn("pendapatan", F.col("unit_terjual") * F.col("harga_satuan"))

print("Tabel Transaksi dari HDFS (5 Baris Pertama):")
df_transaksi.show(5)

# 2. Buat df_target dari dictionary referensi cabang
df_target = spark.createDataFrame(pd.DataFrame(data_target_cabang))
print("Tabel Target Cabang:")
df_target.show()

Tabel Transaksi dari HDFS (5 Baris Pertama):
+--------+--------------------+----------+------------+------------+----------+
|order_id|            kategori|      kota|unit_terjual|harga_satuan|pendapatan|
+--------+--------------------+----------+------------+------------+----------+
|   TRX-0|   Makanan & Minuman| Purworejo|           8|       75000|    600000|
|   TRX-1|          Elektronik|      Solo|           9|       75000|    675000|
|   TRX-2|Kesehatan & Kecan...|      Solo|           6|      100000|    600000|
|   TRX-3|             Fashion|Yogyakarta|           6|      100000|    600000|
|   TRX-4|          Elektronik|Yogyakarta|           2|       75000|    150000|
+--------+--------------------+----------+------------+------------+----------+
only showing top 5 rows

Tabel Target Cabang:
+----------+--------------+----------+
|      kota|target_bulanan|pic_cabang|
+----------+--------------+----------+
|  Magelang|      45000000|      Rani|
|Yogyakarta|      60000000|      

---
### A. Join & Perbandingan Target

In [7]:
# Meringkas total pendapatan per kota dari df_transaksi
df_pendapatan_kota = df_transaksi.groupBy("kota").agg(
    F.sum("pendapatan").alias("total_pendapatan")
)

# Inner Join dengan df_target berdasarkan kota
# Tambahkan kolom pencapaian_persen dan urutkan dari pencapaian tertinggi
df_pencapaian = df_pendapatan_kota.join(df_target, on="kota", how="inner") \
    .withColumn(
        "pencapaian_persen", 
        F.round((F.col("total_pendapatan") / F.col("target_bulanan")) * 100, 2)
    ) \
    .orderBy(F.col("pencapaian_persen").desc())

print("Hasil Analisis Bagian A (Pencapaian Target Cabang):")
df_pencapaian.show()

Hasil Analisis Bagian A (Pencapaian Target Cabang):
+----------+----------------+--------------+----------+-----------------+
|      kota|total_pendapatan|target_bulanan|pic_cabang|pencapaian_persen|
+----------+----------------+--------------+----------+-----------------+
| Purworejo|        45650000|      30000000|     Fitri|           152.17|
|      Solo|        33475000|      40000000|      Bayu|            83.69|
|Yogyakarta|        47275000|      60000000|      Joko|            78.79|
|  Magelang|        31650000|      45000000|      Rani|            70.33|
|  Semarang|        38175000|      55000000|      Sari|            69.41|
+----------+----------------+--------------+----------+-----------------+



---
### B. Window Function — Kategori Terlaris per Kota

In [8]:
# Ringkas total pendapatan per pasangan kota & kategori (best practice: agregasi dulu)
df_kota_kategori = df_transaksi.groupBy("kota", "kategori").agg(
    F.sum("pendapatan").alias("total_pendapatan_kategori")
)

# Definisikan window spesifikasi: partisi berdasarkan kota, urutkan pendapatan menurun
window_kota = Window.partitionBy("kota").orderBy(F.col("total_pendapatan_kategori").desc())

# Tambahkan row_number() dan filter peringkat 1
df_kategori_terlaris = df_kota_kategori \
    .withColumn("peringkat", F.row_number().over(window_kota)) \
    .filter(F.col("peringkat") == 1) \
    .orderBy(F.col("total_pendapatan_kategori").desc())

print("Hasil Analisis Bagian B (Kategori Terlaris per Cabang Kota):")
df_kategori_terlaris.show()

Hasil Analisis Bagian B (Kategori Terlaris per Cabang Kota):
+----------+--------------------+-------------------------+---------+
|      kota|            kategori|total_pendapatan_kategori|peringkat|
+----------+--------------------+-------------------------+---------+
|Yogyakarta|             Fashion|                 13325000|        1|
|  Semarang|        Rumah Tangga|                 11125000|        1|
| Purworejo|Kesehatan & Kecan...|                 10075000|        1|
|      Solo|Kesehatan & Kecan...|                  8425000|        1|
|  Magelang|Kesehatan & Kecan...|                  7275000|        1|
+----------+--------------------+-------------------------+---------+



---
### C. Spark SQL — Jumlah Transaksi per Cabang

In [12]:
# Daftarkan DataFrame sebagai Temporary View
df_transaksi.createOrReplaceTempView("transaksi")
df_target.createOrReplaceTempView("target")

# Jalankan kueri Spark SQL murni
df_sql = spark.sql("""
    SELECT 
        t.kota,
        tg.pic_cabang,
        COUNT(t.order_id) AS jumlah_transaksi
    FROM transaksi t
    JOIN target tg ON t.kota = tg.kota
    GROUP BY t.kota, tg.pic_cabang
    ORDER BY jumlah_transaksi DESC
""")

print("Hasil Analisis Bagian C (Spark SQL):")
df_sql.show()

Hasil Analisis Bagian C (Spark SQL):
+----------+----------+----------------+
|      kota|pic_cabang|jumlah_transaksi|
+----------+----------+----------------+
| Purworejo|     Fitri|             116|
|Yogyakarta|      Joko|             110|
|      Solo|      Bayu|              95|
|  Semarang|      Sari|              93|
|  Magelang|      Rani|              86|
+----------+----------+----------------+



---
### D. Kesimpulan Analisis Bisnis (Bobot 25%)

Berdasarkan hasil pengolahan data transaksi dan target bulanan pada bagian A, B, dan C, berikut kesimpulannya:

1. **Cabang dengan Kinerja Paling Baik (Purworejo):**  
   Cabang **Purworejo** (PIC Fitri) adalah cabang yang paling sukses. Purworejo berhasil melampaui target penjualannya hingga **152,17%**, dengan total pendapatan sebesar **Rp45.650.000** (target awalnya hanya Rp30.000.000). Purworejo juga menjadi cabang yang paling ramai pembeli dengan total **116 transaksi**, dan produk yang paling banyak menyumbang pemasukan di kota ini adalah kategori *Kesehatan & Kecantikan* sebesar **Rp10.075.000**.

2. **Cabang yang Paling Perlu Diperhatikan (Semarang dan Magelang):**  
   Dua cabang yang penjualannya belum memuaskan adalah Semarang (PIC Sari) dan Magelang (PIC Rani) karena keduanya tidak berhasil mencapai target (di bawah 100%):
   - **Semarang:** Memiliki persentase capaian paling rendah di antara semua cabang, yaitu hanya **69,41%**. Pendapatan yang diperoleh sebesar Rp38.175.000 padahal targetnya cukup tinggi yaitu Rp55.000.000, dengan total 93 transaksi (produk terlarisnya adalah *Rumah Tangga* sebesar Rp11.125.000).
   - **Magelang:** Capaian penjualannya hanya **70,33%** (pendapatan Rp31.650.000 dari target Rp45.000.000) dan menjadi cabang yang transaksinya paling sepi, yaitu hanya **86 transaksi**.

3. **Saran / Rekomendasi:**  
   Untuk cabang Semarang, manajemen sebaiknya menurunkan target penjualannya agar lebih realistis dan tidak terlalu membebani tim cabang. Untuk cabang Magelang, sebaiknya diadakan promo atau diskon khusus agar toko menjadi lebih ramai pembeli karena jumlah transaksinya paling sedikit. Selain itu, Semarang juga bisa mencoba memperbanyak stok barang *Kesehatan & Kecantikan* yang terbukti sangat laku keras di Purworejo.

---
### Menutup SparkSession
Menutup sesi Spark untuk membebaskan alokasi CPU thread dan memori JVM:

In [7]:
spark.stop()
print("SparkSession berhasil dihentikan. Resource telah dibebaskan.")

SparkSession berhasil dihentikan. Resource telah dibebaskan.
